# Report figures for the Delta Lakehouse project

This notebook generates **every figure the LaTeX report needs** (17 files) from the `lakehouse-opt`
repository (branch `testing`). Each figure is saved under the exact file name the report expects, so
dropping the files into the report's `figures/` folder replaces the grey placeholder boxes.

**How it works**

- Most figures are drawn from the **evidence files the pipeline already committed** (`docs/evidence/`,
  `docs/profile/`, `docs/benchmark/`, `docs/source_data_manifest.json`). They work on a fresh checkout.
- Four figures are better with the **live Delta tables** in `data/lakehouse/` (after
  `python lakehouse_pipeline.py` and `--steps benchmark`). When Spark or the tables are missing, those
  cells fall back to the evidence files or skip with a clear message; nothing else is affected.
- Numbers are never typed by hand: every label is computed from a file, and each saved figure prints
  its source.

**Run it** from the repository root (or any sub-folder) in the project's Python 3.12 environment:
`jupyter lab report_figures.ipynb`, then *Run All*. Figures go to `report/figures/`
(override with the `REPORT_FIG_DIR` environment variable).

**Before the final run**, regenerate the stale evidence (profile, full pipeline run) so the figures
match the report's numbers.

## 0. Setup

In [3]:
import json, os, re, sys, math, textwrap, traceback
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import numpy as np
import pandas as pd

def find_repo_root(start: Path = Path.cwd()) -> Path:
    for p in [start, *start.parents]:
        if (p / "lakehouse_pipeline.py").exists() and (p / "src").is_dir():
            return p
    raise FileNotFoundError("Run this notebook from inside the lakehouse-opt repository.")

REPO = find_repo_root()
DOCS = REPO / "docs"
LAKE = Path(os.environ.get("LAKEHOUSE_DIR", REPO / "data" / "lakehouse"))
FIG_DIR = Path(os.environ.get("REPORT_FIG_DIR", REPO / "report" / "figures"))
FIG_DIR.mkdir(parents=True, exist_ok=True)

USE_SPARK = True      # set to False to skip every cell that needs Spark
FUNNEL_MONTH = "2025-03"
DPI = 200

C = dict(bronze="#B0703A", silver="#7D8791", gold="#C99A1A", delta="#1F5FA8", accent="#C0392B",
         ok="#2E8B57", grey="#9AA0A6", light="#EEF2F6", text="#1F2328", pale="#F6F8FA")
plt.rcParams.update({
    "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": False,
    "figure.facecolor": "white", "savefig.facecolor": "white",
})

EXPECTED = [
    "lakehouse_layers.png", "delta_log_commit.png", "occ_sequence.png", "medallion_flow.png",
    "file_minmax_grid.png", "pipeline_dataflow.png", "bronze_history.png", "profile_null_block.png",
    "silver_funnel_2025_03.png", "total_mismatch_gaps.png", "cdc_before_after.png",
    "schema_evolution_commits.png", "silver_history.png", "version_timeline.png", "cbd_yoy.png",
    "benchmark_files_scanned.png", "spark_ui_scan_q1.png",
]
RESULTS = {}

class SkipFigure(Exception):
    """Raised when a figure cannot be built from the available inputs."""

def figure(name):
    """Run the decorated function immediately; record created / skipped / failed."""
    def run(fn):
        try:
            fn()
        except SkipFigure as e:
            RESULTS[name] = ("skipped", str(e)); print(f"SKIPPED {name}: {e}")
        except Exception as e:
            RESULTS[name] = ("failed", repr(e)); traceback.print_exc()
        return fn
    return run

def save(fig, name, source):
    path = FIG_DIR / name
    fig.savefig(path, dpi=DPI, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    RESULTS[name] = ("created", source)
    print(f"saved {path}  |  source: {source}")

print("repo      :", REPO)
print("figures to:", FIG_DIR)
print("lakehouse :", LAKE, "(exists)" if LAKE.exists() else "(not found: live-table figures will fall back)")

ModuleNotFoundError: No module named 'matplotlib'

### Load the evidence files

In [ ]:
def load_json(rel):
    p = REPO / rel
    return json.loads(p.read_text()) if p.exists() else None

E = {name: load_json(rel) for name, rel in {
    "manifest":   "docs/source_data_manifest.json",
    "profile":    "docs/profile/bronze_profile.json",
    "bronze_h":   "docs/evidence/bronze_history.json",
    "silver_h":   "docs/evidence/silver_history.json",
    "findings":   "docs/evidence/findings.json",
    "cdc":        "docs/evidence/cdc_merge.json",
    "schema":     "docs/evidence/schema_evolution.json",
    "tt":         "docs/evidence/time_travel.json",
    "vacuum":     "docs/evidence/vacuum_demo.json",
    "gold":       "docs/evidence/gold_verification.json",
    "cbd_zones":  "docs/evidence/cbd_zone_validation.json",
    "bench":      "docs/benchmark/benchmark_results.json",
}.items()}
missing = [k for k, v in E.items() if v is None]
print("loaded :", [k for k, v in E.items() if v is not None])
print("missing:", missing or "none")

def need(*keys):
    gone = [k for k in keys if E.get(k) is None]
    if gone:
        raise SkipFigure(f"missing evidence: {gone}")

def meta(entry):
    """Parse the userMetadata JSON of a history entry."""
    try:
        return json.loads(entry.get("userMetadata") or "{}")
    except json.JSONDecodeError:
        return {}

def metric(entry, key, default=0):
    v = (entry.get("operationMetrics") or {}).get(key, default)
    return int(v) if str(v).lstrip("-").isdigit() else default

### Optional: Spark session (only for the live-table figures)

In [ ]:
spark = None
if USE_SPARK and LAKE.exists():
    try:
        if str(REPO) not in sys.path:
            sys.path.insert(0, str(REPO))
        from src.spark_session import create_spark
        spark = create_spark("report-figures")
        print("Spark", spark.version, "ready")
    except Exception as e:  # pyspark / Java not available in this kernel
        print("Spark not available, live-table cells will fall back:", e)
else:
    print("Spark disabled or no data/lakehouse folder: live-table cells will fall back.")

### Drawing helpers

In [ ]:
def box(ax, x, y, w, h, text, fc="white", ec=None, fontsize=9.5, bold=False, color=None, lw=1.2, style="round,pad=0.02,rounding_size=0.08"):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle=style, fc=fc, ec=ec or C["text"], lw=lw))
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=fontsize,
            weight="bold" if bold else "normal", color=color or C["text"], wrap=True)

def arrow(ax, p1, p2, color=None, ls="-", lw=1.3, text=None, text_offset=(0, 0.12), fontsize=8.5, rad=0.0):
    ax.add_patch(FancyArrowPatch(p1, p2, arrowstyle="-|>", mutation_scale=12, color=color or C["text"],
                                 lw=lw, ls=ls, connectionstyle=f"arc3,rad={rad}"))
    if text:
        mx, my = (p1[0] + p2[0]) / 2 + text_offset[0], (p1[1] + p2[1]) / 2 + text_offset[1]
        ax.text(mx, my, text, ha="center", va="bottom", fontsize=fontsize, color=color or C["text"])

def canvas(w, h, xlim, ylim):
    fig, ax = plt.subplots(figsize=(w, h))
    ax.set_xlim(*xlim); ax.set_ylim(*ylim); ax.axis("off")
    return fig, ax

def table_figure(df, title, note=None, numeric_cols=(), highlight_rows=(), col_widths=None, fontsize=9, width=None):
    nrows, ncols = df.shape
    fig, ax = plt.subplots(figsize=(width or min(13, 1.7 * ncols + 1.5), 0.55 + 0.34 * (nrows + 1)))
    ax.axis("off")
    tbl = ax.table(cellText=df.values, colLabels=list(df.columns), cellLoc="left", colLoc="left",
                   colWidths=col_widths, bbox=[0, 0, 1, 1])
    tbl.auto_set_font_size(False); tbl.set_fontsize(fontsize)
    num_idx = {list(df.columns).index(c) for c in numeric_cols}
    for (r, c), cell in tbl.get_celld().items():
        cell.set_edgecolor("#D0D7DE"); cell.set_linewidth(0.6)
        if c in num_idx:
            cell._loc = "right"; cell.get_text().set_horizontalalignment("right")
        if r == 0:
            cell.set_facecolor(C["light"]); cell.set_text_props(weight="bold")
        elif (r - 1) in highlight_rows:
            cell.set_facecolor("#FFF4E5")
    ax.set_title(title, pad=12)
    if note:
        fig.text(0.0, -0.02, note, fontsize=8, color="#57606A", va="top")
    return fig

def fmt(n):
    return f"{int(n):,}"

def footnote(fig, text, space=0.1):
    """Reserve space at the bottom of the figure and write a source/caveat note there."""
    fig.tight_layout(rect=[0, space, 1, 1])
    fig.text(0.005, 0.01, text, fontsize=8, color="#57606A", va="bottom", wrap=True)

## Part A figures
### 1. `lakehouse_layers.png` — where the table format sits (report §2)

In [ ]:
@figure("lakehouse_layers.png")
def _():
    fig, ax = canvas(10.5, 5.4, (0, 10.5), (0, 6.2))
    engines = ["BI / SQL", "Reports", "Data science", "Machine learning", "Streaming"]
    for i, e in enumerate(engines):
        x = 0.35 + i * 1.98
        box(ax, x, 4.75, 1.78, 0.8, e)
        arrow(ax, (x + 0.89, 4.72), (x + 0.89, 4.0))
    box(ax, 0.35, 2.55, 9.7, 1.4,
        "Transactional metadata layer: Delta Lake  (_delta_log/)\n"
        "ACID commits  |  versions and time travel\nschema enforcement and evolution  |  file statistics for skipping",
        fc="#E7F0FA", ec=C["delta"], bold=True, fontsize=10)
    arrow(ax, (5.2, 2.52), (5.2, 1.88))
    box(ax, 0.35, 0.45, 9.7, 1.4,
        "Open data files: Parquet on object storage (S3 / ADLS / GCS) or HDFS\n"
        "immutable files, readable directly by any engine",
        fc=C["pale"], ec=C["grey"], fontsize=10)
    ax.text(5.35, 2.2, "the log decides which files form each version", fontsize=8, color=C["delta"], va="center")
    ax.text(0.35, 0.1, "Adapted from Armbrust et al., CIDR 2021, Fig. 2. In this project: PySpark 4.0.1 + Delta Lake 4.0.1 on a local file system.",
            fontsize=7.5, color="#57606A")
    save(fig, "lakehouse_layers.png", "diagram")

### 2. `delta_log_commit.png` — a real commit, annotated (report §3)
Prefers a live commit (Z-ORDERed benchmark table, then Bronze v3); falls back to `docs/evidence/delta_log_annotated.md`.

In [ ]:
def read_commit(path):
    return [json.loads(l) for l in Path(path).read_text().splitlines() if l.strip()]

def md_commit_actions(path):
    blocks = [json.loads(b) for b in re.findall(r"```json\n(.*?)\n```", Path(path).read_text(), re.S)]
    actions = []
    for b in blocks:
        if set(b) & {"commitInfo", "add", "remove", "metaData", "protocol"}:
            actions.append(b)
        elif "operation" in b:
            actions.append({"commitInfo": b})
        elif "path" in b:
            actions.append({"add": b})
    return actions

def pick_commit():
    candidates = []
    zlog = LAKE / "benchmark" / "zorder_1col" / "_delta_log"
    if zlog.exists():
        for j in sorted(zlog.glob("*.json"), reverse=True):
            acts = read_commit(j)
            if any("add" in a for a in acts):
                candidates.append((j, acts, "benchmark/zorder_1col")); break
    blog = LAKE / "bronze" / "yellow_trips" / "_delta_log" / f"{3:020d}.json"
    if blog.exists():
        candidates.append((blog, read_commit(blog), "bronze/yellow_trips"))
    ann = DOCS / "evidence" / "delta_log_annotated.md"
    if ann.exists():
        candidates.append((ann, md_commit_actions(ann), "evidence: delta_log_annotated.md"))
    if not candidates:
        raise SkipFigure("no Delta log and no delta_log_annotated.md")
    return candidates[0]

def stats_line(add, cols=("PULocationID", "DOLocationID", "tpep_pickup_datetime")):
    try:
        st = json.loads(add.get("stats") or "{}")
    except json.JSONDecodeError:
        return "stats: (unparsed)"
    parts = [f"numRecords={st.get('numRecords', '?'):,}" if isinstance(st.get("numRecords"), int) else "numRecords=?"]
    for c in cols:
        if c in st.get("minValues", {}):
            parts.append(f"{c} [{st['minValues'][c]} .. {st['maxValues'].get(c)}]")
    return "stats: " + ",  ".join(parts)

@figure("delta_log_commit.png")
def _():
    path, actions, label = pick_commit()
    counts = {}
    for a in actions:
        k = next(iter(a)); counts[k] = counts.get(k, 0) + 1
    lines = [(f"source: {label}", "#57606A", None), ("", None, None)]
    ci = next((a["commitInfo"] for a in actions if "commitInfo" in a), {})
    if ci:
        lines.append(("commitInfo", C["delta"], "bold"))
        params = ci.get("operationParameters", {})
        lines.append((f"  operation={ci.get('operation')}  mode={params.get('mode', '-')}  readVersion={ci.get('readVersion', '-')}", C["text"], None))
        lines.append((f"  isolationLevel={ci.get('isolationLevel', '-')}  isBlindAppend={ci.get('isBlindAppend', '-')}", C["text"], None))
        m = ci.get("operationMetrics", {})
        if m:
            keep = {k: m[k] for k in list(m)[:4]}
            lines.append(("  metrics: " + ", ".join(f"{k}={v}" for k, v in keep.items()), C["text"], None))
        um = ci.get("userMetadata")
        if um:
            lines.append(("  userMetadata: " + textwrap.shorten(um, 88), "#57606A", None))
    md_ = next((a["metaData"] for a in actions if "metaData" in a), None)
    if md_:
        n = len(json.loads(md_["schemaString"])["fields"])
        lines += [("metaData", C["ok"], "bold"), (f"  schema with {n} fields; partitionColumns={md_.get('partitionColumns')}", C["text"], None)]
    adds = [a["add"] for a in actions if "add" in a]
    if adds:
        lines.append((f"add  x{len(adds)}", C["ok"], "bold"))
        for a in adds[:3]:
            lines.append(("  " + textwrap.shorten(a["path"], 80), C["text"], None))
            for part in stats_line(a).split(",  "):
                lines.append(("      " + part, "#57606A", None))
    rem = counts.get("remove", 0)
    lines.append((f"remove  x{rem}" + ("   (tombstones: files leave the snapshot, stay on disk until VACUUM)" if rem else ""), C["accent"], "bold"))
    fig, ax = plt.subplots(figsize=(10.5, 0.24 * len(lines) + 1.2)); ax.axis("off")
    for i, (t, col, w) in enumerate(lines):
        ax.text(0, 1 - i / len(lines), t, family="monospace", fontsize=9.5, color=col or C["text"],
                weight=w or "normal", va="top", transform=ax.transAxes)
    ax.set_title("One Delta commit: each line of the JSON file is an action", pad=10)
    footnote(fig, "add.stats (row count, min/max, null counts) is what data skipping reads at planning time.", space=0.08)
    save(fig, "delta_log_commit.png", str(Path(path).relative_to(REPO)) if REPO in Path(path).parents else str(path))

### 3. `occ_sequence.png` — optimistic concurrency control (report §3.5)

In [ ]:
@figure("occ_sequence.png")
def _():
    fig, ax = canvas(10.5, 6.6, (0, 10.5), (0, 10))
    lanes = {"Writer A": 1.6, "_delta_log/": 5.25, "Writer B": 8.9}
    for name, x in lanes.items():
        box(ax, x - 1.0, 9.1, 2.0, 0.6, name, fc="#E7F0FA" if name == "_delta_log/" else "white",
            ec=C["delta"] if name == "_delta_log/" else C["text"], bold=True)
        ax.plot([x, x], [4.3, 9.05], color=C["grey"], lw=1, ls=(0, (4, 3)))
    A, L, B = lanes.values()
    arrow(ax, (L, 8.4), (A, 8.4), text="read snapshot v7", color=C["text"])
    arrow(ax, (L, 8.4), (B, 8.4), text="read snapshot v7", color=C["text"])
    ax.text(A, 7.55, "write new Parquet files\n(invisible: no commit yet)", ha="center", fontsize=8.5)
    ax.text(B, 7.55, "write new Parquet files\n(invisible: no commit yet)", ha="center", fontsize=8.5)
    arrow(ax, (A, 6.6), (L, 6.6), text="create 000...08.json  ->  OK", color=C["ok"])
    arrow(ax, (B, 5.7), (L, 5.7), text="create 000...08.json  ->  exists", color=C["accent"])
    arrow(ax, (L, 4.9), (B, 4.9), text="read commit v8: what changed?", color=C["text"])
    box(ax, 4.0, 1.9, 2.9, 1.7, "No conflict\n(blind append, or files and\npartitions B did not read)\n-> commit 000...09.json", fc="#EAF5EE", ec=C["ok"], fontsize=8.5)
    box(ax, 7.4, 1.9, 2.9, 1.7, "Conflict\n(a file B read was removed,\nor new files match B's predicate)\n-> Concurrent...Exception", fc="#FBEDEB", ec=C["accent"], fontsize=8.5)
    arrow(ax, (B - 0.1, 4.55), (5.45, 3.65), color=C["ok"], text="retry, no data rewritten", text_offset=(-0.4, 0.05))
    arrow(ax, (B, 4.55), (B - 0.05, 3.65), color=C["accent"])
    ax.text(0.15, 2.75, "No locks. Data files are written first;\nthe only contested step is creating\nthe next numbered commit file,\nwhich the storage allows only once.\n\nIn this project Bronze appends are\nblind appends (isBlindAppend=true),\nso they never conflict: the pipeline\nassumes a single writer.",
            fontsize=8.5, color="#57606A", va="center")
    save(fig, "occ_sequence.png", "diagram (Delta concurrency-control docs)")

### 4. `medallion_flow.png` — layers, partition keys and row counts (report §5)

In [ ]:
def flow_numbers():
    need("manifest", "bronze_h", "tt", "silver_h", "gold")
    raw_rows = sum(f["rows"] for f in E["manifest"]["trip_files"].values())
    n_files = len(E["manifest"]["trip_files"])
    bronze_rows = sum(metric(h, "numOutputRows") for h in E["bronze_h"])
    silver_rows = E["tt"]["silver_rows_per_version"][-1]["rows"]
    updated = sum(metric(h, "numTargetRowsUpdated") for h in E["silver_h"])
    rejected = bronze_rows - silver_rows - updated     # conservation: Bronze = rejected + inserted + updated
    return dict(raw_rows=raw_rows, n_files=n_files, bronze_rows=bronze_rows, silver_rows=silver_rows,
                updated=updated, rejected=rejected, gold_trips=E["gold"]["gold_trips"],
                zone_rows=E["gold"]["zone_hourly_rows"])

@figure("medallion_flow.png")
def _():
    n = flow_numbers()
    fig, ax = canvas(12, 5.2, (0, 12), (0, 6))
    box(ax, 0.2, 3.3, 2.1, 1.6, f"Raw files\n{n['n_files']} TLC Parquet\n{fmt(n['raw_rows'])} rows", fc=C["pale"], ec=C["grey"])
    box(ax, 2.9, 3.3, 2.5, 1.6, f"Bronze\nappend-only, as received\n{fmt(n['bronze_rows'])} rows\npartition: _source_month", fc="#F6E9DE", ec=C["bronze"])
    box(ax, 6.1, 4.1, 2.7, 1.4, f"Silver trips\ntyped, deduplicated, flagged\n{fmt(n['silver_rows'])} rows\npartition: pickup_month", fc="#EEF0F2", ec=C["silver"])
    box(ax, 6.1, 1.9, 2.7, 1.4, f"Quarantine\noriginal values + reason\n{fmt(n['rejected'])} rows\npartition: _source_month", fc="#FBEDEB", ec=C["accent"])
    box(ax, 9.5, 3.3, 2.3, 1.6, f"Gold marts\n{fmt(n['gold_trips'])} trips\nzone x date x hour:\n{fmt(n['zone_rows'])} rows", fc="#FBF3DC", ec=C["gold"])
    arrow(ax, (2.3, 4.1), (2.9, 4.1)); arrow(ax, (5.4, 4.3), (6.1, 4.7)); arrow(ax, (5.4, 3.9), (6.1, 2.7))
    arrow(ax, (8.8, 4.8), (9.5, 4.3))
    ax.text(6.0, 0.95, f"Conservation: {fmt(n['bronze_rows'])} Bronze = {fmt(n['rejected'])} rejected + "
            f"{fmt(n['silver_rows'])} inserted + {fmt(n['updated'])} updated", ha="center", fontsize=9, weight="bold")
    ax.text(6.0, 0.35, "A trip picked up on 31 Jan inside the February file: Bronze _source_month=2024-02, "
            "Silver pickup_month=2024-01.", ha="center", fontsize=8.5, color="#57606A")
    save(fig, "medallion_flow.png", "manifest, bronze/silver history, time_travel.json, gold_verification.json")

### 5. `file_minmax_grid.png` — per-file min/max before and after Z-ORDER (report §6.4)
Reads `add.stats` straight from the benchmark tables' `_delta_log` (pure Python, no Spark). Needs `data/lakehouse/benchmark/` from `python lakehouse_pipeline.py --steps benchmark`.

In [ ]:
def active_files(table: Path):
    """Replay a Delta log (checkpoint + JSON commits) and return {path: add_action} of the latest snapshot."""
    log = table / "_delta_log"
    if not log.exists():
        raise SkipFigure(f"{table.relative_to(REPO) if REPO in table.parents else table} has no _delta_log")
    files, start = {}, -1
    last = log / "_last_checkpoint"
    if last.exists():
        import pyarrow.parquet as pq
        v = json.loads(last.read_text())["version"]
        for part in sorted(log.glob(f"{v:020d}.checkpoint*.parquet")):
            for row in pq.read_table(part, columns=["add"]).to_pylist():
                if row.get("add"):
                    files[row["add"]["path"]] = row["add"]
        start = v
    for j in sorted(log.glob("[0-9]*.json")):
        if int(j.stem) <= start:
            continue
        for act in read_commit(j):
            if "add" in act:
                files[act["add"]["path"]] = act["add"]
            elif "remove" in act:
                files.pop(act["remove"]["path"], None)
    return files

def ranges(table: Path, col="PULocationID"):
    out = []
    for a in active_files(table).values():
        st = json.loads(a.get("stats") or "{}")
        if col in st.get("minValues", {}):
            out.append((st["minValues"][col], st["maxValues"][col]))
    if not out:
        raise SkipFigure(f"no {col} statistics in {table.name}")
    return sorted(out)

@figure("file_minmax_grid.png")
def _():
    target = 132
    panels = [("baseline (random order)", LAKE / "benchmark" / "baseline"),
              ("zorder_1col: ZORDER BY (PULocationID)", LAKE / "benchmark" / "zorder_1col")]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
    for ax, (title, path) in zip(axes, panels):
        rs = ranges(path)
        hit = [lo <= target <= hi for lo, hi in rs]
        lw = max(1.0, min(4.0, 300 / len(rs)))
        for i, ((lo, hi), h) in enumerate(zip(rs, hit)):
            ax.plot([i, i], [lo, hi], color=C["accent"] if h else C["grey"], lw=lw, alpha=0.9 if h else 0.6,
                    solid_capstyle="butt")
        ax.axhline(target, color=C["delta"], lw=1, ls="--")
        ax.set_title(f"{title}\n{sum(hit)} of {len(rs)} files may contain zone {target}", fontsize=10.5)
        ax.set_xlabel("files (sorted by min PULocationID)")
    axes[0].set_ylabel("PULocationID range [min, max] per file")
    axes[1].text(len(rs) * 0.98, target + 6, f"WHERE PULocationID = {target}", ha="right", color=C["delta"], fontsize=8.5)
    fig.suptitle("Data skipping reads only files whose [min, max] contains the value", x=0.01, ha="left", weight="bold", y=0.99)
    fig.tight_layout(rect=[0, 0, 1, 0.94])
    save(fig, "file_minmax_grid.png", "add.stats in data/lakehouse/benchmark/{baseline,zorder_1col}/_delta_log")

## Part B figures
### 6. `pipeline_dataflow.png` — the ten steps with their key numbers (report §7)

In [ ]:
@figure("pipeline_dataflow.png")
def _():
    n = flow_numbers()
    need("profile", "cdc", "schema", "tt", "vacuum", "bench")
    q1 = E["bench"]["summary"]["per_query"]["q1_zone"]
    steps = [
        ("1 Download", f"{n['n_files']} files, SHA-256\n{fmt(n['raw_rows'])} rows"),
        ("2 Reference", "zone lookup + CBD list\n-> dim_zone"),
        ("3 Bronze", f"{len(E['bronze_h'])} append commits\n{fmt(n['bronze_rows'])} rows"),
        ("4 Profile", f"{E['profile']['keys']['keys_seen_in_more_than_one_file']} key in 2 files\nno table written"),
        ("5 Silver", f"{fmt(n['silver_rows'])} rows\n{fmt(n['rejected'])} quarantined"),
        ("6 CDC", f"{E['cdc']['merge_metrics']['updated']} updated\n{E['cdc']['merge_metrics']['inserted']} inserted"),
        ("7 Evolution", f"+{', '.join(E['schema']['bronze']['evolution_event']['added'])}\nat v{E['schema']['bronze']['evolution_event']['version']}"),
        ("8 Time travel", f"{len(E['tt']['silver_rows_per_version'])} Silver versions\nVACUUM broke v{', v'.join(map(str, E['vacuum']['broken_versions']))}"),
        ("9 Gold", f"{fmt(n['gold_trips'])} trips\n4 marts"),
        ("10 Benchmark", f"JFK query: {q1['baseline']['files_scanned']} -> {q1['zorder_1col']['files_scanned']} files"),
    ]
    fig, ax = canvas(13, 4.9, (0, 13), (0, 5.2))
    layer_col = {3: C["bronze"], 5: C["silver"], 6: C["silver"], 9: C["gold"]}
    pos = {}
    for i, (title, body) in enumerate(steps):
        row = 0 if i < 5 else 1
        col = i if i < 5 else 9 - i            # second row runs right-to-left (snake)
        x, y = 0.2 + col * 2.58, 3.3 - row * 2.6
        pos[i] = (x, y)
        ec = layer_col.get(i + 1, C["text"])
        box(ax, x, y, 2.2, 1.6, f"{title}\n\n{body}", ec=ec, lw=1.6 if i + 1 in layer_col else 1.1, fontsize=8.8)
    for i in range(9):
        (x1, y1), (x2, y2) = pos[i], pos[i + 1]
        if i == 4:
            arrow(ax, (x1 + 1.1, y1), (x2 + 1.1, y2 + 1.6))
        elif i < 4:
            arrow(ax, (x1 + 2.2, y1 + 0.8), (x2, y2 + 0.8))
        else:
            arrow(ax, (x1, y1 + 0.8), (x2 + 2.2, y2 + 0.8))
    (xc, yc), (xb, yb) = pos[5], pos[2]
    arrow(ax, (xc + 0.3, yc + 1.6), (xb + 1.9, yb), ls="--", color=C["accent"], rad=0.25,
          text="CDC feed lands in Bronze (v6),\nthen runs through Silver again", text_offset=(-0.2, -0.35))
    save(fig, "pipeline_dataflow.png", "evidence files (see cell)")

### 7. `bronze_history.png` — Bronze commit history (report §8)

In [ ]:
@figure("bronze_history.png")
def _():
    need("bronze_h")
    rows = []
    for h in sorted(E["bronze_h"], key=lambda r: r["version"]):
        m = meta(h)
        rows.append([f"v{h['version']}", h["operation"], (h.get("operationParameters") or {}).get("mode", "-"),
                     m.get("batch_id", "-"), metric(h, "numFiles"), fmt(metric(h, "numOutputRows")),
                     f"{metric(h, 'numOutputBytes') / 1e6:,.1f}" if metric(h, 'numOutputBytes') >= 1e5 else f"{metric(h, 'numOutputBytes') / 1e6:.2f}"])
    df = pd.DataFrame(rows, columns=["Version", "Operation", "Mode", "Batch (userMetadata)", "Files", "Rows", "MB written"])
    total = sum(metric(h, "numOutputRows") for h in E["bronze_h"])
    synthetic = [i for i, r in enumerate(rows) if not str(r[3]).startswith("yellow_")]
    fig = table_figure(df, f"Bronze: {len(rows)} append-only commits, {total:,} rows",
                       note="Every commit is WRITE / Append; userMetadata names the source batch. Highlighted: synthetic CDC feed.",
                       numeric_cols=("Files", "Rows", "MB written"), highlight_rows=synthetic,
                       col_widths=[0.08, 0.1, 0.08, 0.34, 0.07, 0.17, 0.12], width=11)
    save(fig, "bronze_history.png", "docs/evidence/bronze_history.json")

### 8. `profile_null_block.png` — the NULL block is the `payment_type = 0` stream (report §8.3)

In [ ]:
@figure("profile_null_block.png")
def _():
    need("profile")
    P = E["profile"]; months = sorted(P["values"])
    rows = np.array([P["values"][m]["rows"] for m in months])
    pt0 = np.array([P["distributions"]["payment_type"][m].get("0", 0) for m in months])
    cols = ["passenger_count", "RatecodeID", "congestion_surcharge", "store_and_fwd_flag", "Airport_fee"]
    cols = [c for c in cols if f"null__{c}" in P["values"][months[0]]]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12.5, 4.2), gridspec_kw={"width_ratios": [1, 1.6]})
    share = 100 * pt0 / rows
    a1.bar(months, share, color=[C["bronze"] if m < "2025" else C["accent"] for m in months])
    for i, v in enumerate(share):
        a1.text(i, v + 0.4, f"{v:.1f}%", ha="center", fontsize=8.5)
    a1.set_title("payment_type = 0 share of trips"); a1.set_ylabel("% of rows"); a1.tick_params(axis="x", rotation=45)
    width = 0.8 / len(cols); x = np.arange(len(months))
    for k, c in enumerate(cols):
        a2.bar(x + k * width - 0.4 + width / 2, [P["values"][m][f"null__{c}"] for m in months], width, label=f"NULL {c}", alpha=0.85)
    a2.plot(x, pt0, "k_", markersize=26, mew=2.2, label="payment_type = 0 rows")
    a2.set_xticks(x, months, rotation=45); a2.set_title("NULL counts equal the payment_type = 0 count")
    a2.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v/1e3:,.0f}k")); a2.legend(fontsize=7.5, frameon=False, ncol=2)
    synth = [h["version"] for h in E.get("bronze_h") or [] if not meta(h).get("batch_id", "").startswith("yellow_")]
    note = f"Profile of Bronze v{P.get('bronze_version')}."
    if synth and P.get("bronze_version", -1) >= min(synth):
        note += " That version includes the 30-row synthetic CDC batch (counted under 2025-03; negligible here)."
    footnote(fig, note, space=0.08)
    save(fig, "profile_null_block.png", "docs/profile/bronze_profile.json")

#### Optional extra: missingness correlation heatmap (needs Spark and Bronze)
Not a report slot; useful for the profiling slide. Uses a deterministic 1% sample.

In [ ]:
@figure("profile_missingness_heatmap.png")
def _():
    if spark is None:
        raise SkipFigure("Spark not available")
    from pyspark.sql import functions as F
    bronze = spark.read.format("delta").load(str(LAKE / "bronze" / "yellow_trips"))
    cols = [c for c in bronze.columns if not c.startswith("_")]
    sample = bronze.sample(fraction=0.01, seed=42).select([F.col(c).isNull().cast("int").alias(c) for c in cols]).toPandas()
    sample = sample.loc[:, sample.std() > 0]
    corr = sample.corr()
    fig, ax = plt.subplots(figsize=(7.5, 6.2))
    im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(range(len(corr)), corr.columns, rotation=60, ha="right", fontsize=8)
    ax.set_yticks(range(len(corr)), corr.columns, fontsize=8)
    for i in range(len(corr)):
        for j in range(len(corr)):
            ax.text(j, i, f"{corr.iat[i, j]:.2f}", ha="center", va="center", fontsize=7)
    fig.colorbar(im, ax=ax, shrink=0.8); ax.set_title("Correlation of NULL indicators (1% sample of Bronze)")
    save(fig, "profile_missingness_heatmap.png", "Bronze Delta table, 1% sample (seed 42)")

### 9. `silver_funnel_2025_03.png` — how one batch splits (report §9)
Exact reasons come from `ops/silver_batch_log` when Spark and the tables are available; otherwise from the profile's rule dry run, corrected so the total matches the Silver history.

In [ ]:
def batch_numbers(month):
    need("bronze_h", "silver_h")
    b = next((h for h in E["bronze_h"] if meta(h).get("batch_id", "").startswith(f"yellow_{month}_")), None)
    s = next((h for h in E["silver_h"] if meta(h).get("batch_id", "").startswith(f"yellow_{month}_")), None)
    if b is None or s is None:
        raise SkipFigure(f"no batch for {month} in the histories")
    bronze = metric(b, "numOutputRows")
    inserted = metric(s, "numTargetRowsInserted") if s["operation"] == "MERGE" else metric(s, "numOutputRows")
    updated = metric(s, "numTargetRowsUpdated")
    return meta(b)["batch_id"], bronze, inserted, updated, s["version"]

def reject_reasons(batch_id, month, rejected):
    log = LAKE / "ops" / "silver_batch_log"
    if spark is not None and (log / "_delta_log").exists():
        row = spark.read.format("delta").load(str(log)).filter(f"batch_id = '{batch_id}'").collect()
        if row:
            return json.loads(row[0]["reject_reason_counts"]), "ops/silver_batch_log"
    need("profile")
    prev = dict(E["profile"]["rule_preview"]["reject_reason"].get(month, {}))
    extra = sum(prev.values()) - rejected
    if extra > 0 and prev.get("KEY_COLLISION", 0) >= extra:
        prev["KEY_COLLISION"] -= extra   # the profile ran after the CDC batch landed with _source_month = this month
    return prev, "profile rule dry run (corrected to the Silver total)"

@figure("silver_funnel_2025_03.png")
def _():
    month = FUNNEL_MONTH
    batch_id, bronze, inserted, updated, version = batch_numbers(month)
    rejected = bronze - inserted - updated
    reasons, src = reject_reasons(batch_id, month, rejected)
    reasons = dict(sorted(((k, v) for k, v in reasons.items() if v), key=lambda kv: -kv[1]))
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12.5, 3.9), gridspec_kw={"width_ratios": [1.25, 1]})
    a1.barh(["Bronze rows"], [bronze], color=C["bronze"])
    a1.barh(["Silver"], [inserted + updated], color=C["silver"], label="accepted (MERGE)")
    a1.barh(["Silver"], [rejected], left=[inserted + updated], color=C["accent"], label="quarantined")
    a1.text(bronze, 0, f" {fmt(bronze)}", va="center", fontsize=9)
    a1.text((inserted + updated) / 2, 1, f"{fmt(inserted + updated)} accepted", va="center", ha="center", color="white", fontsize=9, weight="bold")
    a1.text(bronze, 1, f" {fmt(rejected)} rejected ({100 * rejected / bronze:.2f}%)", va="center", fontsize=9, color=C["accent"])
    a1.set_xlim(0, bronze * 1.35); a1.set_title(f"Batch {batch_id} -> Silver v{version}"); a1.xaxis.set_visible(False)
    a1.spines["bottom"].set_visible(False); a1.invert_yaxis()
    names, vals = list(reasons), list(reasons.values())
    a2.barh(names, vals, color=C["accent"]); a2.set_xscale("log"); a2.invert_yaxis()
    for i, v in enumerate(vals):
        a2.text(v * 1.15, i, fmt(v), va="center", fontsize=8.5)
    a2.set_title("Reject reasons (log scale)"); a2.set_xlim(1, max(vals) * 12)
    footnote(fig, f"Reasons from {src}. Bronze = accepted + rejected is checked for every batch.", space=0.08)
    save(fig, f"silver_funnel_{month.replace('-', '_')}.png", f"bronze/silver history + {src}")

### 10. `total_mismatch_gaps.png` — why the TOTAL_MISMATCH rule was redesigned (report §9.4)

In [ ]:
SURCHARGE_GAPS = {-3.25, -2.5, -0.75, 2.5}

@figure("total_mismatch_gaps.png")
def _():
    need("findings")
    tm = E["findings"]["total_mismatch"]
    total = sum(r["mismatched_rows"] for r in tm["per_month"])
    gaps = sorted((float(g["gap"]), g["count"]) for g in tm["most_common_gaps"])
    share = 100 * sum(c for g, c in gaps if g in SURCHARGE_GAPS) / total
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.3), gridspec_kw={"width_ratios": [1.5, 1]})
    xs = [f"{g:+.2f}" for g, _ in gaps]
    a1.bar(xs, [c for _, c in gaps], color=[C["accent"] if g in SURCHARGE_GAPS else C["grey"] for g, _ in gaps])
    a1.set_yscale("log"); a1.set_ylabel("rows (log scale)"); a1.set_xlabel("total_amount - sum(components)")
    a1.tick_params(axis="x", rotation=60)
    a1.set_title(f"{share:.1f}% of {total:,} mismatches equal a surcharge amount", fontsize=11)
    by = pd.DataFrame(tm["most_common_gaps_by_year"])
    piv = by.pivot_table(index="gap", columns="year", values="count", aggfunc="sum").fillna(0).sort_index()
    x = np.arange(len(piv)); w = 0.4
    for k, (yr, col) in enumerate(zip(piv.columns, [C["bronze"], C["delta"]])):
        a2.bar(x + (k - 0.5) * w, piv[yr], w, label=yr, color=col)
    a2.set_xticks(x, [f"{g:+.2f}" for g in piv.index], rotation=60); a2.set_yscale("log")
    a2.set_title("By year: -2.50 (2024) becomes -3.25 (2025)", fontsize=11); a2.legend(frameon=False)
    footnote(fig, "Red: congestion-surcharge amounts (2.50; 0.75 CBD fee; 3.25 = both). The rule now checks total_amount "
             "against a range [low, high] instead of an exact sum.", space=0.1)
    save(fig, "total_mismatch_gaps.png", "docs/evidence/findings.json (diagnostic on the original rule)")

### 11. `cdc_before_after.png` — the correction MERGE, row by row (report §10)

In [ ]:
@figure("cdc_before_after.png")
def _():
    need("cdc")
    c = E["cdc"]; rows = []
    for s in c["samples"]:
        b, a = s.get("before"), s["after"]
        kind = "insert" if not b else ("tip + fare" if b["fare_amount"] != a["fare_amount"] else "tip only")
        f = lambda k: f"{b[k]:.2f} -> {a[k]:.2f}" if b else f"- -> {a[k]:.2f}"
        rows.append([s["trip_key"][:12] + "...", kind, f("fare_amount"), f("tip_amount"), f("total_amount"),
                     f"{b['_batch_id'] if b else '-'} -> {a['_batch_id']}"])
    df = pd.DataFrame(rows, columns=["trip_key", "Change", "Fare", "Tip", "Total", "_batch_id"])
    p, m = c["planned"], c["merge_metrics"]
    title = (f"Silver v{c['silver_version_before']} -> v{c['silver_version_after']}: planned {p['updates']} updates / "
             f"{p['inserts']} inserts, Delta reported {m['updated']} / {m['inserted']}")
    fig = table_figure(df, title, width=14, fontsize=8.5, col_widths=[0.12, 0.08, 0.12, 0.11, 0.13, 0.44],
                       note=f"Synthetic feed '{c['batch_id']}' (seed {c['seed']}) built from real {c['target_month']} trips. "
                            f"Real monthly files produced {c['real_cdc_from_monthly_files']['cross_file_updates']} cross-file updates.")
    save(fig, "cdc_before_after.png", "docs/evidence/cdc_merge.json")

### 12. `schema_evolution_commits.png` — the schema change in the log (report §11)

In [ ]:
@figure("schema_evolution_commits.png")
def _():
    need("schema")
    S = E["schema"]; col = S["column"]
    before = next(c["columns"] for c in S["bronze"]["schema_changes"] if c["event"] == "CREATE")
    ev = S["bronze"]["evolution_event"]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12.5, 4.8), gridspec_kw={"width_ratios": [1.1, 1]})
    a1.axis("off")
    a1.set_title(f"Bronze metaData: v0 has {len(before)} fields, v{ev['version']} ({ev['operation']}) has {len(before) + len(ev['added'])}")
    fields = before + ev["added"]; half = math.ceil(len(fields) / 2)
    for i, f in enumerate(fields):
        new = f in ev["added"]
        a1.text(0.02 + 0.5 * (i // half), 0.93 - 0.07 * (i % half), ("+ " if new else "  ") + f,
                family="monospace", fontsize=9, color=C["ok"] if new else C["text"], weight="bold" if new else "normal",
                transform=a1.transAxes)
    events = "   ".join(f"{t}: v{S[t]['evolution_event']['version']} {S[t]['evolution_event']['operation']}"
                        for t in ("bronze", "silver", "silver_rejected"))
    a1.text(0.02, -0.04, events, fontsize=8.5, color="#57606A", transform=a1.transAxes)
    nc = S["silver_null_check"]; months = sorted(nc)
    pct = [100 * nc[m]["null"] / nc[m]["rows"] for m in months]
    a2.bar(months, pct, color=[C["grey"] if p > 50 else C["delta"] for p in pct])
    for i, (m, p) in enumerate(zip(months, pct)):
        a2.text(i, p + 2, f"{p:.0f}%", ha="center", fontsize=9)
    a2.set_ylim(0, 115); a2.set_ylabel(f"% NULL {col}"); a2.tick_params(axis="x", rotation=45)
    a2.set_title("NULL = column absent in the source file; 0 = no fee")
    save(fig, "schema_evolution_commits.png", "docs/evidence/schema_evolution.json")

### 13. `silver_history.png` — Silver `DESCRIBE HISTORY` with MERGE metrics (report §12)

In [ ]:
@figure("silver_history.png")
def _():
    need("silver_h")
    rows = []
    for h in sorted(E["silver_h"], key=lambda r: r["version"]):
        merge = h["operation"] == "MERGE"
        rows.append([f"v{h['version']}", h["operation"], meta(h).get("batch_id", "-"),
                     fmt(metric(h, "numTargetRowsInserted") if merge else metric(h, "numOutputRows")),
                     fmt(metric(h, "numTargetRowsUpdated")),
                     metric(h, "numTargetFilesAdded") if merge else metric(h, "numFiles"),
                     metric(h, "numTargetFilesRemoved"), fmt(metric(h, "numTargetRowsCopied")),
                     f"{metric(h, 'executionTimeMs') / 1000:.1f}" if merge else "-"])
    df = pd.DataFrame(rows, columns=["Version", "Operation", "Batch", "Inserted", "Updated", "Files added",
                                     "Files removed", "Rows copied", "Time (s)"])
    cow = [i for i, r in enumerate(rows) if r[6]]
    fig = table_figure(df, "Silver history: one WRITE, then one MERGE per batch",
                       note="Monthly MERGEs only add files. The CDC MERGE (highlighted) rewrote one file: copy-on-write.",
                       numeric_cols=("Inserted", "Updated", "Files added", "Files removed", "Rows copied", "Time (s)"),
                       highlight_rows=cow, width=12.5,
                       col_widths=[0.06, 0.08, 0.2, 0.1, 0.08, 0.1, 0.12, 0.12, 0.08], fontsize=8.5)
    save(fig, "silver_history.png", "docs/evidence/silver_history.json")

### 14. `version_timeline.png` — time travel and the VACUUM result (report §12.2)

In [ ]:
@figure("version_timeline.png")
def _():
    need("tt", "silver_h", "vacuum")
    versions = E["tt"]["silver_rows_per_version"]
    hist = {h["version"]: h for h in E["silver_h"]}
    broken = set(E["vacuum"]["broken_versions"])
    fig, ax = canvas(12.5, 3.9, (-0.7, len(versions) - 0.3), (-1.9, 2.3))
    ax.plot([0, len(versions) - 1], [0, 0], color=C["grey"], lw=2, zorder=0)
    prev_cols = None
    for v in versions:
        i = v["version"]; h = hist.get(i, {})
        bad = i in broken
        ax.scatter([i], [0], s=420, color="white" if bad else C["silver"], edgecolor=C["accent"] if bad else C["text"], lw=2, zorder=2)
        ax.text(i, 0, f"v{i}", ha="center", va="center", fontsize=9, weight="bold", zorder=3)
        if bad:
            ax.plot([i - 0.22, i + 0.22], [-0.3, 0.3], color=C["accent"], lw=2.5, zorder=4)
        batch = meta(h).get("batch_id", "").replace("yellow_", "").rsplit("_", 1)[0] or "-"
        ax.text(i, 0.55, f"{v['operation']}\n{batch}", ha="center", fontsize=8.5)
        ax.text(i, -0.62, f"{v['rows']:,} rows", ha="center", fontsize=8)
        removed = metric(h, "numTargetFilesRemoved")
        if removed:
            ax.text(i, -1.05, f"removed {removed} file", ha="center", fontsize=8, color=C["accent"])
        if prev_cols is not None and v["columns"] != prev_cols:
            ax.text(i, 1.45, f"+1 column ({prev_cols} -> {v['columns']})", ha="center", fontsize=8, color=C["ok"])
        prev_cols = v["columns"]
    ax.text(-0.6, -1.7, f"VACUUM (0 h) on a copy deleted {E['vacuum']['files_listed_by_dry_run']} file; unreadable afterwards: "
            f"{', '.join('v' + str(b) for b in sorted(broken))}. Every other version still reads. "
            f"The real table was never vacuumed (v0 = {E['vacuum']['real_table_oldest_version_rows']:,} rows).",
            fontsize=8.5, color="#57606A")
    ax.set_title("Silver versions: one per batch; only the version pointing at the rewritten file breaks after VACUUM", loc="left")
    save(fig, "version_timeline.png", "time_travel.json, silver_history.json, vacuum_demo.json")

### 15. `cbd_yoy.png` — the congestion question and the CBD list check (report §13)

In [ ]:
@figure("cbd_yoy.png")
def _():
    need("gold", "cbd_zones")
    yoy = pd.DataFrame(E["gold"]["cbd_yoy"])
    flows = [f for f in ["INTO_CBD", "OUT_OF_CBD", "WITHIN_CBD"] if f in set(yoy["cbd_flow"])]
    months = sorted(yoy["month_of_year"].unique())
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.6), gridspec_kw={"width_ratios": [1.3, 1]})
    w = 0.8 / len(flows); x = np.arange(len(months))
    colors = {"INTO_CBD": C["accent"], "OUT_OF_CBD": C["bronze"], "WITHIN_CBD": C["delta"]}
    for k, f in enumerate(flows):
        vals = [yoy[(yoy.cbd_flow == f) & (yoy.month_of_year == m)]["relative_to_non_cbd_pp"].iloc[0] for m in months]
        bars = a1.bar(x + k * w - 0.4 + w / 2, vals, w, label=f, color=colors[f])
        for b, v in zip(bars, vals):
            a1.text(b.get_x() + b.get_width() / 2, v + (0.4 if v >= 0 else -0.4), f"{v:+.1f}", ha="center",
                    va="bottom" if v >= 0 else "top", fontsize=8)
    a1.axhline(0, color=C["text"], lw=0.8)
    non = [yoy[yoy.month_of_year == m]["non_cbd_yoy_pct"].iloc[0] for m in months]
    a1.set_xticks(x, [f"month {m}\n(NON_CBD {n:+.1f}%)" for m, n in zip(months, non)])
    a1.set_ylabel("YoY change vs NON_CBD (pp)"); a1.legend(frameon=False, fontsize=8.5)
    a1.set_title("2025 vs 2024 trips per day, relative to non-CBD trips")
    z = pd.DataFrame(E["cbd_zones"]); z = z[z.verdict != "TOO_FEW_TRIPS"]
    unknown = z.PULocationID.isin([264, 265]) | z.borough.isin(["Unknown", "N/A"]) | z.zone.isin(["N/A", "Unknown"])
    z_unknown, z = z[unknown], z[~unknown]
    groups = [("listed\n(official)", z[(z.is_cbd) & (z.cbd_source != "inferred_geography")], C["delta"]),
              ("listed\n(inferred)", z[(z.is_cbd) & (z.cbd_source == "inferred_geography")], C["ok"]),
              ("not listed", z[~z.is_cbd], C["grey"])]
    rng = np.random.default_rng(42)
    for i, (lab, g, col) in enumerate(groups):
        a2.scatter(i + rng.uniform(-0.18, 0.18, len(g)), 100 * g.fee_share, s=16, color=col, alpha=0.8)
    lo = z[z.is_cbd].sort_values("fee_share").iloc[0]; hi = z[~z.is_cbd].sort_values("fee_share").iloc[-1]
    a2.axhline(100 * lo.fee_share, color=C["delta"], lw=0.8, ls="--"); a2.axhline(100 * hi.fee_share, color=C["grey"], lw=0.8, ls="--")
    a2.text(2.45, 100 * lo.fee_share, f"lowest listed {100 * lo.fee_share:.1f}%\n({lo.zone})", fontsize=7.5, va="center")
    a2.text(2.45, 100 * hi.fee_share, f"highest unlisted {100 * hi.fee_share:.1f}%\n({hi.zone})", fontsize=7.5, va="center")
    a2.set_xticks(range(3), [g[0] for g in groups]); a2.set_xlim(-0.5, 3.6); a2.set_ylabel("post-policy pickups paying the CBD fee (%)")
    a2.set_title(f"CBD list check: {int(z.is_cbd.sum())} listed zones, zones with >= 200 pickups")
    extra = "" if z_unknown.empty else (f" Unknown-zone buckets excluded from the check (e.g. zone "
             f"{int(z_unknown.iloc[0].PULocationID)}: {100 * z_unknown.fee_share.max():.1f}%).")
    footnote(fig, "Descriptive, not causal: January 2025 includes 4 pre-policy days; weekday mix not controlled." + extra, space=0.08)
    save(fig, "cbd_yoy.png", "gold_verification.json (cbd_yoy), cbd_zone_validation.json")

### 16. `benchmark_files_scanned.png` — files scanned per layout (report §14)

In [ ]:
VARIANTS = ["baseline", "compacted", "zorder_1col", "zorder_2col", "liquid_cluster"]
QLABEL = {"q1_zone": "Q1 zone (JFK)", "q2_zone_week": "Q2 zone + week", "q3_control": "Q3 control (cash)"}

@figure("benchmark_files_scanned.png")
def _():
    need("bench")
    per = E["bench"]["summary"]["per_query"]
    queries = [q for q in QLABEL if q in per]
    variants = [v for v in VARIANTS if v in per[queries[0]]]
    vcol = dict(zip(VARIANTS, [C["grey"], "#B8C4CF", C["delta"], "#6FA3D8", C["ok"]]))
    fig, ax = plt.subplots(figsize=(12.5, 4.8))
    w = 0.8 / len(variants); x = np.arange(len(queries))
    for k, v in enumerate(variants):
        files = [per[q][v]["files_scanned"] for q in queries]
        cols = [("#D5D8DC" if q == "q3_control" else vcol[v]) for q in queries]
        bars = ax.bar(x + k * w - 0.4 + w / 2, files, w, color=cols, edgecolor="white", label=v)
        for b, q in zip(bars, queries):
            r = per[q][v]
            ax.text(b.get_x() + b.get_width() / 2, b.get_height() * 1.08, f"{r['files_scanned']}\n{r['median_ms']:.0f} ms",
                    ha="center", fontsize=7.5)
    ax.set_yscale("log"); ax.set_ylim(1, 1000); ax.set_ylabel("files scanned (log scale)")
    ax.set_xticks(x, [QLABEL[q] for q in queries])
    handles = [plt.Rectangle((0, 0), 1, 1, color=vcol[v]) for v in variants]
    ax.legend(handles, variants, frameon=False, ncol=len(variants), loc="upper right", fontsize=8.5)
    b = E["bench"]
    ax.set_title(f"{b['rows']:,} rows, {b['rounds']} rounds, {b['target_file_mb']} MB target files: "
                 "clustering cuts files read; the control query (grey) cannot skip")
    save(fig, "benchmark_files_scanned.png", "docs/benchmark/benchmark_results.json")

### 17. `spark_ui_scan_q1.png` — the scan node before and after Z-ORDER (report §14)
Runs Q1 live on the benchmark tables when possible (same metric code as `optimization_benchmark.py`); otherwise uses the first measured run in `benchmark_results.json`.

In [ ]:
def q1_scan(variant):
    base, zpath = LAKE / "benchmark" / variant, None
    if spark is not None and (base / "_delta_log").exists():
        from optimization_benchmark import QUERIES, collect_scan_metrics
        spark.catalog.clearCache()
        df = QUERIES["q1_zone"][1](spark.read.format("delta").load(str(base)))
        rows = [r.asDict() for r in df.collect()]
        return {**collect_scan_metrics(df), "result": rows}, f"live query on data/lakehouse/benchmark/{variant}"
    need("bench")
    run = next(r for r in E["bench"]["runs"] if r["query"] == "q1_zone" and r["variant"] == variant and not r.get("warmup"))
    return run, "docs/benchmark/benchmark_results.json (first measured run)"

@figure("spark_ui_scan_q1.png")
def _():
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
    sources = set()
    for ax, (variant, ec) in zip(axes, [("baseline", C["grey"]), ("zorder_1col", C["delta"])]):
        m, src = q1_scan(variant); sources.add(src)
        ax.axis("off"); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
        ax.add_patch(FancyBboxPatch((0.02, 0.05), 0.96, 0.8, boxstyle="round,pad=0.01", fc="#FBFCFD", ec=ec, lw=2))
        ax.text(0.06, 0.76, "Scan parquet (Delta)", weight="bold", fontsize=11, va="top")
        trips = m["result"][0].get("trips") if m.get("result") else None
        lines = [("number of files read", f"{m['files_scanned']:,}"),
                 ("size of files read", f"{m['bytes_read'] / 2**20:,.1f} MB"),
                 ("matching trips", f"{trips:,}" if trips is not None else "-")]
        for i, (k, v) in enumerate(lines):
            ax.text(0.06, 0.55 - i * 0.16, k, fontsize=10, color="#57606A")
            ax.text(0.94, 0.55 - i * 0.16, v, fontsize=12, weight="bold", ha="right", color=ec if i == 0 else C["text"])
        ax.set_title(f"Q1  WHERE PULocationID = 132  on  {variant}", fontsize=10.5)
    footnote(fig, "Metrics read from the executed plan's scan node (numFiles, filesSize; MB = 2^20 bytes, as in the benchmark), "
             f"as in the Spark UI SQL tab. Source: {'; '.join(sorted(sources))}.", space=0.1)
    save(fig, "spark_ui_scan_q1.png", "; ".join(sorted(sources)))

## Summary

In [ ]:
summary = pd.DataFrame(
    [(n, *RESULTS.get(n, ("not run", ""))) for n in EXPECTED] +
    [(n, *v) for n, v in RESULTS.items() if n not in EXPECTED],
    columns=["figure", "status", "source / reason"])
print(f"{(summary.status == 'created').sum()} created, {(summary.status == 'skipped').sum()} skipped, "
      f"{(summary.status == 'failed').sum()} failed  ->  {FIG_DIR}")
summary

**Next step:** copy the files from `report/figures/` into the LaTeX project's `figures/` folder (or point
`REPORT_FIG_DIR` at it before running). The report picks them up automatically on the next compile.
Figures marked *skipped* need the live tables: run `python lakehouse_pipeline.py` and
`python lakehouse_pipeline.py --steps benchmark`, then re-run this notebook.